# Probability Trees

Companion notebook for [Probability Trees](../../notes/probability/probability_tree.md).

A tree stores branch probabilities. Multiplying along a path gives the probability of the joint outcome and adding the paths that lead to an event gives its probability. We implement trees as small Python structures, check the rules, and compare with simulation.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import product
from fractions import Fraction

rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (7, 3.5)

## 1. A tiny tree class

Each node has a list of `(label, probability, child)` branches. The function enumerates all paths with their probabilities and checks that branch probabilities at every node sum to one.

In [2]:
def make_tree(*branches):
    return list(branches)

def paths(tree, prefix=(), prob=Fraction(1)):
    if tree is None:
        yield prefix, prob
        return
    assert sum(p for _, p, _ in tree) == 1, "branches must sum to 1"
    for label, p, child in tree:
        yield from paths(child, prefix + (label,), prob * p)

# Coin then die
die = make_tree(*[(str(k), Fraction(1, 6), None) for k in range(1, 7)])
coin_die = make_tree(("H", Fraction(1, 2), die), ("T", Fraction(1, 2), die))
allp = list(paths(coin_die))
print("number of leaves:", len(allp), " total probability:", sum(p for _, p in allp))
print("P(H and even):", sum(p for path, p in allp if path[0] == "H" and int(path[1]) % 2 == 0))

number of leaves: 12  total probability: 1
P(H and even): 1/4


## 2. Drawing marbles without replacement

A bag with 6 red and 4 green marbles, two draws. The second-level probabilities depend on the first draw: $\frac59,\frac49$ after red and $\frac69,\frac39$ after green.

In [3]:
second_after_R = make_tree(("R", Fraction(5, 9), None), ("G", Fraction(4, 9), None))
second_after_G = make_tree(("R", Fraction(6, 9), None), ("G", Fraction(3, 9), None))
marbles = make_tree(("R", Fraction(6, 10), second_after_R), ("G", Fraction(4, 10), second_after_G))
for path, p in paths(marbles):
    print("".join(path), p, round(float(p), 4))
print("sum:", sum(p for _, p in paths(marbles)))

RR 1/3 0.3333
RG 4/15 0.2667
GR 4/15 0.2667
GG 2/15 0.1333
sum: 1


## 3. Verifying with simulation

We simulate the marble draws and compare the frequencies of the four paths with the tree.

In [4]:
bag = np.array(list("RRRRRRGGGG"))
n = 100_000
counts = {}
for _ in range(n):
    d = rng.choice(10, 2, replace=False)
    key = "".join(bag[d])
    counts[key] = counts.get(key, 0) + 1
for key in ("RR", "RG", "GR", "GG"):
    print(key, "simulated:", round(counts[key] / n, 4))

RR simulated: 0.3334
RG simulated: 0.2669
GR simulated: 0.2661
GG simulated: 0.1336


## 4. Diagnostic tree and Bayes

With prevalence 5%, sensitivity 90%, specificity 95%, the four leaves have probabilities $0.045,0.005,0.0475,0.9025$. Then $P(+)=0.0925$ and $P(D\mid+)=0.045/0.0925\approx0.486$ (reverse conditioning from the tree).

In [5]:
leaves = {
    "D,+":  0.05 * 0.90,
    "D,-":  0.05 * 0.10,
    "ND,+": 0.95 * 0.05,
    "ND,-": 0.95 * 0.95,
}
for k, v in leaves.items():
    print(f"{k:5s} {v:.4f}")
p_pos = leaves["D,+"] + leaves["ND,+"]
print("total:", round(sum(leaves.values()), 6))
print("P(+) =", round(p_pos, 4), " P(D|+) =", round(leaves["D,+"] / p_pos, 4))

D,+   0.0450
D,-   0.0050
ND,+  0.0475
ND,-  0.9025
total: 1.0
P(+) = 0.0925  P(D|+) = 0.4865


## 5. A best-of-three series

Team A wins each game with probability 0.6. It wins the series with AA, ABA, or BAA: $0.36+2(0.36)(0.4)=0.648$. We enumerate the tree of three games (playing all three for convenience) and compare.

In [6]:
p = 0.6
total = 0.0
for g in product("AB", repeat=3):
    prob = np.prod([p if x == "A" else 1 - p for x in g])
    if g.count("A") >= 2:
        total += prob
print("exact (3 games played in full):", round(total, 4))

sim = (rng.random((200_000, 3)) < p).sum(axis=1) >= 2
print("simulated:", round(sim.mean(), 4))

exact (3 games played in full): 0.648
simulated: 0.6483
